### SCPT to VS

In [5]:
# Import packages for use:
from SCPT_Func import Master, interleave
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import HTML, Layout, HBox, VBox, Dropdown, FloatText, Button, Textarea
from IPython.display import display
plt.rcParams['font.family'] = 'Times New Roman'
pd.set_option('display.max_columns', None)

### Initialization: 
import data in .csv form and create csv file for output

In [6]:
'''

read travel time data csv as dataframe, 
make sure the last row is not NAN in the .csv file 
and unit of depth is meter and unit of travel time is millisecond.
change the name to the target input file. 

'''

TT_DATA = pd.read_csv('sample_110.csv')

TT_depth = np.asarray(TT_DATA['depth (m)'])
Meas_TT = np.asarray(TT_DATA['traveltime (ms)'])

In [7]:
try:
    VS_SLOPEBREAK_DF = pd.read_csv("SLOPEBREAK_Vs.csv")
except:
    VS_SLOPEBREAK_DF = pd.DataFrame(columns=['Velocity', 'Top_depth', 'Bottom_depth'])
    VS_SLOPEBREAK_DF.to_csv("SLOPEBREAK_Vs.csv", index=False, header=True)

### GUI

In [8]:
%matplotlib widget

# Setup
style = {'description_width': 'initial'}

velocity_updated = []
top_depth_updated = []
bottom_depth_updated = []

VsZ_input = widgets.FloatText(value=1.0, description='VsZ Direct Interpretation', style=style)

button_layout = widgets.Layout(width='300px', height='40px')
EXPORT = Button(description='Export Result', layout = button_layout)
MANUAL_REVISE = Button(description='Manual Revision', layout = button_layout)

# Dynamic checkbox container (empty to start)
checkbox_box = widgets.VBox(layout=Layout(flex_flow='row wrap', width='100%'))

box_layout_2 = widgets.Layout(width='100%', border='solid 2px', padding='20px')
SEARCH = VBox([
    HTML('<center><font size="+1.5"><b>SCPT Travel Time Direct Interpretation Automated Tool</b>'),
    HBox([VsZ_input], layout=Layout(width='100%')),
    HBox([EXPORT, 
          MANUAL_REVISE], layout=Layout(width='100%')),
    checkbox_box  # add the checkboxes below the buttons
], layout=box_layout_2)

out = widgets.Output()

# --- Reactive SCPT update ---
def Update_SCPT(_):

    global velocity_updated
    global top_depth_updated
    global bottom_depth_updated

    out.clear_output()
    with out:

        # Run main computation and get breaks
        velocity, TT, results, top_depth, bottom_depth = Master(TT_depth, Meas_TT, slope_break_method=0)
        breaks = sorted(results.get('candidate_breakpoints', []))
        used_breaks = [str(round(b, 3)) for b in results.get('breaks', [])]
        
        # using subplot to plot qt, fz, Ic, Vs with depth for the selected traveltimeMeta_ID
        fig, axs = plt.subplots(1, 2, figsize=(6, 5), dpi=100)

        axs[0].scatter(TT, TT_depth, label='Travel Time Data', color='black', s=10)

        colors = plt.get_cmap('tab10').colors

        # Plot each segment with a different color
        for i in range(len(results['breaks']) - 1):
            # Define segment depth range
            z_start = results['breaks'][i]
            z_end = results['breaks'][i + 1]

            # Get depths in this segment (new_depths ensures breaks are included)
            segment_mask = (results['new_depths'] >= z_start) & (results['new_depths'] <= z_end)
            segment_depths = results['new_depths'][segment_mask]
            segment_tt = results['fitted_values_for_plot'][segment_mask]

            # Plot segment
            axs[0].plot(segment_tt, segment_depths, color=colors[i], 
                        # label=f'Segment {i+1}'
                        )

        axs[0].set_xlabel('Travel Time (ms)')
        axs[0].set_xlim(0, np.max(TT) * 1.1)
        if np.max(TT) > 150:
            axs[0].set_xticks(np.arange(0, np.max(TT) * 1.1, 50))
        elif np.max(TT) > 90:
            axs[0].set_xticks(np.arange(0, np.max(TT) * 1.1, 30))
        elif np.max(TT) > 50:
            axs[0].set_xticks(np.arange(0, np.max(TT) * 1.1, 20))
        else:
            axs[0].set_xticks(np.arange(0, np.max(TT) * 1.1, 10))
        axs[0].set_title('Travel Time vs Depth')
        axs[0].grid(True, which="both", ls="--")
        axs[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)


        axs[1].plot(interleave(velocity, velocity), interleave(top_depth, bottom_depth), label='Direct Interpreted Velocity', color='blue')

        axs[1].set_xlim(0, 600)
        axs[1].set_xticks(np.arange(0, 601, 100))

        axs[1].set_xlabel('Velocity (m/s)')
        axs[1].set_title('Velocity vs Depth')
        axs[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        axs[1].grid(True, which="both", ls="--")

        # Share common y-axis limits across all subplots
        y_max = np.nanmax(bottom_depth)  # Use the maximum depth from the interpreted model for consistent y-axis limits
        for ax in axs:
            ax.set_ylim(y_max + 1, 0)  # inverted depth axis, same range for all

        plt.subplots_adjust(bottom=0.28, wspace=0.35)
        plt.tight_layout()
        plt.show()

        # Calculate VsZ for direct interpretation
        top_depth_dir = np.asarray(top_depth)
        top_depth_dir[0] = 0.0
        VsZ = np.max(np.asarray(bottom_depth)) / np.sum((np.asarray(bottom_depth) - top_depth_dir) / np.asarray(velocity))

        # Update the checkboxes dynamically
        new_checkboxes = [
            widgets.Checkbox(description=str(round(b, 3)), indent=False, value=(str(round(b, 3)) in used_breaks))
            for b in sorted(breaks)
        ]
        checkbox_box.children = new_checkboxes

        VsZ_input.value = VsZ

        velocity_updated = velocity
        top_depth_updated = top_depth
        bottom_depth_updated = bottom_depth

def Full_Manual(_):

    global velocity_updated
    global top_depth_updated
    global bottom_depth_updated

    out.clear_output()
    with out:
        # Get selected breakpoints
        selected_breakpoints = [float(checkbox.description) for checkbox in checkbox_box.children if checkbox.value]
        if not selected_breakpoints:
            print("No breakpoints selected.")
            return
        # Run full manual regression
        velocity, TT, results, top_depth, bottom_depth = Master(TT_depth, Meas_TT, slope_break_method=1, breakpoints=selected_breakpoints)

        # using subplot to plot qt, fz, Ic, Vs with depth for the selected traveltimeMeta_ID
        fig, axs = plt.subplots(1, 2, figsize=(6, 5), dpi=100)

        axs[0].scatter(TT, TT_depth, label='Travel Time Data', color='black', s=10)

        colors = plt.get_cmap('tab10').colors

        # Plot each segment with a different color
        for i in range(len(results['breaks']) - 1):
            # Define segment depth range
            z_start = results['breaks'][i]
            z_end = results['breaks'][i + 1]

            # Get depths in this segment (new_depths ensures breaks are included)
            segment_mask = (results['new_depths'] >= z_start) & (results['new_depths'] <= z_end)
            segment_depths = results['new_depths'][segment_mask]
            segment_tt = results['fitted_values_for_plot'][segment_mask]

            # Plot segment
            axs[0].plot(segment_tt, segment_depths, color=colors[i], 
                        # label=f'Segment {i+1}'
                        )

        axs[0].set_xlabel('Travel Time (ms)')
        if np.max(TT) > 150:
            axs[0].set_xticks(np.arange(0, np.max(TT) * 1.1, 50))
        elif np.max(TT) > 90:
            axs[0].set_xticks(np.arange(0, np.max(TT) * 1.1, 30))
        elif np.max(TT) > 50:
            axs[0].set_xticks(np.arange(0, np.max(TT) * 1.1, 20))
        else:
            axs[0].set_xticks(np.arange(0, np.max(TT) * 1.1, 10))
        axs[0].set_title('Travel Time vs Depth')
        axs[0].grid(True, which="both", ls="--")
        axs[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

        axs[1].plot(interleave(velocity, velocity), interleave(top_depth, bottom_depth), label='Direct Interpreted Velocity', color='blue')

        axs[1].set_xlim(0, 600)
        axs[1].set_xticks(np.arange(0, 601, 100))

        axs[1].set_xlabel('Velocity (m/s)')
        axs[1].set_title('Velocity vs Depth')
        axs[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        axs[1].grid(True, which="both", ls="--")

        # Share common y-axis limits across all subplots
        y_max = np.nanmax(bottom_depth)
        for ax in axs:
            ax.set_ylim(y_max + 1, 0)  # inverted depth axis, same range for all

        plt.subplots_adjust(bottom=0.28, wspace=0.35)
        plt.tight_layout()
        plt.show()

        # Calculate VsZ for direct interpretation
        top_depth_dir = np.asarray(top_depth)
        top_depth_dir[0] = 0.0
        VsZ = np.max(np.asarray(bottom_depth)) / np.sum((np.asarray(bottom_depth) - top_depth_dir) / np.asarray(velocity))
        # print(f"Directly Interpreted VsZ: {VsZ:.2f} m/s to depth {np.max(np.asarray(bottom_depth)):.2f} m")

        VsZ_input.value = VsZ

        velocity_updated = velocity
        top_depth_updated = top_depth
        bottom_depth_updated = bottom_depth

# Export Function
def Export (_):
    
    global velocity_updated
    global top_depth_updated
    global bottom_depth_updated

    # Save to CSV
    try:
        VS_SLOPEBREAK_DF = pd.read_csv('SLOPEBREAK_Vs.csv')
    except FileNotFoundError:
        VS_SLOPEBREAK_DF = pd.DataFrame(columns=['travelTimeMeta_ID', 'Velocity', 'Top_depth', 'Bottom_depth'])

    velocity_save = velocity_updated
    top_depth_save = top_depth_updated
    bottom_depth_save = bottom_depth_updated

    # Save the Vs profile to a CSV file
    new_data2 = pd.DataFrame({
    'Velocity': velocity_save,
    'Top_depth': top_depth_save,
    'Bottom_depth': bottom_depth_save})
    VS_SLOPEBREAK_DF = pd.concat([VS_SLOPEBREAK_DF, new_data2], ignore_index=True)
    # Save to CSV
    VS_SLOPEBREAK_DF.to_csv('SLOPEBREAK_Vs.csv', index=False)

# Initial display
display(SEARCH)
display(out)

# Trigger first update manually
Update_SCPT(None)

# Attach trigger to dropdown
EXPORT.on_click(Export)
MANUAL_REVISE.on_click(Full_Manual)


Output()